# W03 : Data Contract — Content Refresh / Opportunity Scoring Lane

**Lane:** Lane 2 : Refresh / Content Opportunity Scoring  
**Week 2 framing:** rank pages by how much they look like a recent-decline pattern  
**Data:** FlyRank internship warehouse on Hugging Face (`FlyRank/internship-warehouse`)

> Skills loaded: `writing-data-contracts/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

## Setup : install DuckDB and authenticate

In [11]:
# Install DuckDB (only needed in Colab)
!pip install duckdb --quiet

In [12]:
import duckdb
import pandas as pd
from google.colab import userdata

# Load HF token from Colab secrets (never paste a token in a cell : repo is public)
HF_TOKEN = userdata.get('HF_TOKEN')

# DuckDB connection with Hugging Face secret
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# Base path for the warehouse tables
BASE = "hf://datasets/FlyRank/internship-warehouse"

# Quick sanity check : should print a row count close to what the data dictionary says
result = con.execute(f"""
    SELECT COUNT(*) AS row_count
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("Row count for month=2026-03:", result['row_count'][0])

Row count for month=2026-03: 9841378


---
## 1. The Contract : five plain answers

### What one row means
One row = **one content item (a web page) on one report date, for one client**.  
In `fact_content_daily_performance`, the grain is `(report_date, client_hash_id, content_hash_id)`.  
For feature building I collapse the daily rows into a per-page summary over the first 14 days of the month.  
The label is computed from the last 14 days of the same month.

### Tables I will use
| Table | Why |
|---|---|
| `fact_content_daily_performance` | daily GSC + GA4 signals per page : source of all features and the label |
| `dim_clients` | to check `gsc_data_start` and `ga4_data_start` per client before setting any window |
| `dim_content` | to get content metadata (type, word count) for richer features later |

### Time window
I iterate on **`month=2026-03`** (March 2026) : a mid-panel month, not the final month.  
The final month (`month=2026-06`) is sealed: it is the natural outcome window and must not be used for label logic or feature development.

### What I predict (label / proxy)
I predict `is_declining` = **a proxy label, not a real future outcome**.  
A page gets `is_declining = 1` when its impressions in the **last 14 days** of the month are more than 20% below its impressions in the **first 14 days** of that same month.  
This is a proxy because it measures a decline that already happened inside the window : it does not tell me whether a content refresh will actually help the page recover.

### One thing I deliberately exclude
**Rows where `client_has_ga4 = FALSE` are excluded from GA4 features.**  
Before a client's `ga4_data_start` date, GA4 columns are zero-filled by the warehouse pipeline.  
Those zeros mean "no data", not "no traffic". Using them as real engagement signals would be wrong.

---
## 2. Field Classification

Every field I plan to touch goes in exactly one bucket.

### Features : knowable before the decision moment
| Field | Why it is safe |
|---|---|
| `gsc_impressions` (sum, first 14d) | search exposure already recorded in the past window |
| `gsc_avg_position` (avg, first 14d) | search rank in the past -- does not depend on the label window |
| `gsc_clicks` (sum, first 14d) | click count from the past -- fully in the feature window |
| `ctr` (clicks / impressions x 100, first 14d) | calculated strictly from past clicks and impressions |
| `active_days` (count of days with impressions > 0, first 14d) | consistency signal -- counts only past days |

### Label / proxy -- the thing I predict (NEVER a feature)
| Field | Rule |
|---|---|
| `is_declining` (computed) | 1 when last-14d impressions < 0.8 x first-14d impressions |
| `gsc_impressions` (last 14d) | raw input to the label -- if used as a feature, that is leakage |

### Context -- for grouping / splitting only, not for the model
| Field | Role |
|---|---|
| `report_date` | used to split rows into feature window vs label window |
| `client_hash_id` | used for grouped train/test splits -- never a model input |
| `content_hash_id` | unique key for joining -- never a model input |
| `month` (partition key) | selects the right data slice -- never a model input |

### Excluded -- with a reason for each
| Field | Why excluded |
|---|---|
| All GA4 columns when `client_has_ga4 = FALSE` | zeros are fill-values from the pipeline, not real engagement |
| `provider_used`, `model_used` | product-decision columns -- the model should not learn from which LLM wrote the article |
| Last-14d GSC columns (other than for the label) | future information relative to the feature window -- using them is leakage |

---
## 3. Prove three facts with three queries

All three queries run on `month=2026-03` (the mid-panel iteration month).  
No guesses : every contract claim above has a query to back it.

### Query 1 : Grain check

One row really is one `(report_date, client_hash_id, content_hash_id)`.  
Zero rows returned = grain holds.

In [13]:
grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate grain rows found: {len(grain_check)}")
print("Expected: 0  (zero rows = grain holds -- one row per date-client-page)")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0
Expected: 0  (zero rows = grain holds -- one row per date-client-page)


,report_date,client_hash_id,content_hash_id,c


### Query 2 : Row count and date span

How many rows, unique pages, and unique clients are in March 2026?  
What are the first and last report dates?

In [14]:
counts = con.execute(f"""
    SELECT
        COUNT(*)                               AS total_rows,
        COUNT(DISTINCT client_hash_id)         AS unique_clients,
        COUNT(DISTINCT content_hash_id)        AS unique_pages,
        MIN(report_date)                       AS earliest_date,
        MAX(report_date)                       AS latest_date
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("=== March 2026 slice ===")
counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== March 2026 slice ===


,total_rows,unique_clients,unique_pages,earliest_date,latest_date
0,9841378,55,331437,2026-03-01,2026-03-31


### Query 3 : Availability check (`client_has_ga4 IS TRUE`)

How many rows have real GA4 data vs fill-value zeros?  
Only rows where `client_has_ga4 IS TRUE` are safe to use GA4 metrics.

In [15]:
availability = con.execute(f"""
    SELECT
        client_has_ga4,
        COUNT(*)                         AS row_count,
        COUNT(DISTINCT client_hash_id)   AS clients,
        COUNT(DISTINCT content_hash_id)  AS pages
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_has_ga4
    ORDER BY client_has_ga4 DESC
""").df()

total = availability['row_count'].sum()
safe  = availability.loc[availability['client_has_ga4'] == True, 'row_count']
safe_count = int(safe.values[0]) if len(safe) > 0 else 0

print("=== GA4 availability breakdown ===")
print(f"Rows where client_has_ga4 IS TRUE : {safe_count:,}  ({safe_count/total*100:.1f}%) -- safe to use GA4 columns")
print(f"Total rows: {total:,}")
print()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== GA4 availability breakdown ===
Rows where client_has_ga4 IS TRUE : 6,822,637  (69.3%) -- safe to use GA4 columns
Total rows: 9,841,378



,client_has_ga4,row_count,clients,pages
0,True,6822637,43,260737
1,False,3018741,22,166412


---
## 4a. Five Features

Features are computed from the **first 14 days of March 2026** only : the feature window.  
The label comes from the **last 14 days of March 2026** : a separate, later window.

| Feature | What it measures | Knowable at decision moment because... |
|---|---|---|
| `impressions_first14` | total GSC impressions in days 1-14 | search exposure already recorded in the past 14 days before the decision moment |
| `avg_position_first14` | mean search rank in days 1-14 | search rank in the past 14 days, fully known before the decision moment |
| `clicks_first14` | total clicks in days 1-14 | search clicks already recorded in days 1-14, known before the label window begins |
| `ctr_first14` | clicks / impressions x 100, days 1-14 | click-through rate in the past 14 days, calculated only from past clicks and impressions |
| `active_days_first14` | number of days with at least 1 impression in days 1-14 | count of days with search visibility in days 1-14, entirely in the past |

In [16]:
# Build the feature frame for March 2026
# Feature window : first 14 days (2026-03-01 to 2026-03-14)
# Label window   : last 14 days  (2026-03-18 to 2026-03-31)  [3 day gap as buffer]

feature_frame = con.execute(f"""
    WITH
    feat AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)                               AS impressions_first14,
            AVG(NULLIF(gsc_avg_position, 0))                   AS avg_position_first14,
            SUM(gsc_clicks)                                    AS clicks_first14,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
                ELSE NULL
            END                                                AS ctr_first14,
            COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END)    AS active_days_first14
        FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-14'
        GROUP BY client_hash_id, content_hash_id
    ),
    lbl AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_last14
        FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date BETWEEN '2026-03-18' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        feat.client_hash_id,
        feat.content_hash_id,
        feat.impressions_first14,
        ROUND(feat.avg_position_first14, 2)       AS avg_position_first14,
        feat.clicks_first14,
        ROUND(feat.ctr_first14, 4)                AS ctr_first14,
        feat.active_days_first14,
        -- Label: 1 if last-14d impressions dropped more than 20% vs first-14d
        CASE
            WHEN feat.impressions_first14 > 0
             AND lbl.impressions_last14 < feat.impressions_first14 * 0.8
            THEN 1
            ELSE 0
        END                                        AS is_declining
    FROM feat
    JOIN lbl
      ON feat.client_hash_id  = lbl.client_hash_id
     AND feat.content_hash_id = lbl.content_hash_id
    WHERE feat.impressions_first14 > 0
""").df()

print(f"Feature frame: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]} columns")
print(f"Unique clients : {feature_frame['client_hash_id'].nunique()}")
print(f"Unique pages   : {feature_frame['content_hash_id'].nunique():,}")
print(f"Label rate (is_declining=1): {feature_frame['is_declining'].mean()*100:.1f}%")
feature_frame.head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 150,443 rows x 8 columns
Unique clients : 44
Unique pages   : 150,443
Label rate (is_declining=1): 38.1%


,client_hash_id,content_hash_id,impressions_first14,avg_position_first14,clicks_first14,ctr_first14,active_days_first14,is_declining
0,client_3ffa76342f366962,content_bc8fdefd9b3cc94b,1.0,NaN,0.0,0.0,1,0
1,client_3ffa76342f366962,content_c55f0cfffb312dea,4.0,3.75,0.0,0.0,4,1
2,client_3ffa76342f366962,content_422014fd6575b359,2.0,5.50,0.0,0.0,2,1
3,client_3ffa76342f366962,content_714aa574c5ef054e,7.0,6.10,0.0,0.0,6,1
4,client_3ffa76342f366962,content_0202977651fb9ac7,1.0,9.00,0.0,0.0,1,1


---
## 4b. The Trap : deliberate label leakage (then removed)

The leakage lesson from notebook 02: if you include a column that comes from the label, the model sees the answer during training  
The score jumps toward perfect --> and the model is useless on new data

**The trap:** add `impressions_last14` as a feature
This is the raw input to `is_declining`. The model almost perfectly predicts the label -- because it already has the answer

In [17]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Get the last-14d impressions column (the leaky column)
lbl_raw = con.execute(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_last14
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-18' AND '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
""").df()

df_leaky = feature_frame.merge(lbl_raw, on=['client_hash_id', 'content_hash_id'], how='left')
df_leaky = df_leaky.dropna(subset=[
    'impressions_first14', 'avg_position_first14', 'clicks_first14',
    'ctr_first14', 'active_days_first14', 'impressions_last14'
])

# LEAKY features : impressions_last14 is the answer in disguise
LEAKY_FEATURES = [
    'impressions_first14', 'avg_position_first14', 'clicks_first14',
    'ctr_first14', 'active_days_first14',
    'impressions_last14'   # <-- THE LEAKY COLUMN
]

X_leaky = df_leaky[LEAKY_FEATURES].fillna(0).values
y_leaky  = df_leaky['is_declining'].values

leaky_scores = cross_val_score(
    GradientBoostingClassifier(n_estimators=50, random_state=42),
    X_leaky, y_leaky, cv=3, scoring='roc_auc'
)

print("=== LEAKY model (impressions_last14 included) ===")
print(f"ROC-AUC: {leaky_scores.mean():.3f}  <-- looks great, but it is cheating")
print("The model already has the answer -- last-14d impressions IS what defines is_declining")

=== LEAKY model (impressions_last14 included) ===
ROC-AUC: 0.991  <-- looks great, but it is cheating
The model already has the answer -- last-14d impressions IS what defines is_declining


In [18]:
# HONEST model : leaky column removed
HONEST_FEATURES = [
    'impressions_first14', 'avg_position_first14', 'clicks_first14',
    'ctr_first14', 'active_days_first14'
]

df_clean = feature_frame.dropna(subset=HONEST_FEATURES + ['is_declining'])

X_honest = df_clean[HONEST_FEATURES].fillna(0).values
y_honest  = df_clean['is_declining'].values

honest_scores = cross_val_score(
    GradientBoostingClassifier(n_estimators=50, random_state=42),
    X_honest, y_honest, cv=3, scoring='roc_auc'
)

print("=== HONEST model (leaky column removed) ===")
print(f"ROC-AUC: {honest_scores.mean():.3f}  <-- real performance")
print()
print(f"Score dropped from {leaky_scores.mean():.3f} (leaky) to {honest_scores.mean():.3f} (honest).")
print("Lesson: the leaky column was not a feature : it was the label in disguise.")
print("The honest number is what we can actually expect on new, unseen data.")

=== HONEST model (leaky column removed) ===
ROC-AUC: 0.613  <-- real performance

Score dropped from 0.991 (leaky) to 0.613 (honest).
Lesson: the leaky column was not a feature : it was the label in disguise.
The honest number is what we can actually expect on new, unseen data.


---
## 5. One Named Limitation of This Slice

**The panel is unbalanced : not every client has the same history depth**

Some clients in the warehouse have 17 months of data; others have 3 months or fewer
In March 2026, pages from new or short-history clients may not have any data in the feature window (days 1-14) even though they appear in other months

If this is ignored, the model trains mostly on long-history clients and will perform poorly on newer clients : which may be exactly the clients where good content scoring matters most

The right fix: always check `dim_clients.gsc_data_start` before setting any feature window, and report model performance separately for short-history vs long-history clients

---
## Self-check

- [x] Five plain-words contract answers: row meaning, tables used, time window, label/proxy, one deliberate exclusion
- [x] Exactly three verification queries with outputs visible -- grain (`HAVING COUNT > 1`), counts + date span, availability (`IS TRUE`)
- [x] Five-feature frame built from first 14 days, each feature has an "available when?" line
- [x] Deliberate-leak experiment shown (leaky ROC-AUC printed), column then removed, honest number kept
- [x] One named limitation: unbalanced panel -- client history depth differs
- [x] No client names, URLs, or private queries anywhere
- [x] Notebook runs top to bottom with no errors (Runtime -> Run all)